# Trimmed-Away Excess Inflation Contribution

This notebook classifies trimmed-away PCE components using the Shapiro demand/supply/ambiguous labels, computes monthly excess contributions relative to trimmed mean PCE inflation, and reconciles those classified gaps to official headline PCE inflation from FRED `PCEPI`.

The component-universe headline is reconstructed from the 178 trimmed-mean replication components. The official headline is calculated from the FRED `PCEPI` index. Their difference is kept as an explicit `coverage_residual_gap` rather than being assigned to demand, supply, or ambiguous.

The notebook includes two 12-month views:

- rolling 12-month sums of monthly factor gaps
- exact Shapley allocations of the compounded official headline-minus-trimmed gap using four factors: demand, supply, ambiguous, and coverage residual

Decimal-rate columns are the source of truth. Percentage-point helper columns are added only for 12-month outputs.


In [1]:
from itertools import permutations
from pathlib import Path
import json
import math
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

INPUTS = {
    'trimmed_away': Path('outputs/trimmed_away_components.csv'),
    'classifications': Path('output/shapiro_supercore_decomp/pce_shapiro_line_classifications.csv'),
    'trimmed_rates': Path('outputs/trimmed_mean_rates.csv'),
    'component_audit': Path('outputs/component_monthly_audit.csv'),
}

OUTPUT_CLASSIFIED = Path('outputs/trimmed_away_components_classified.csv')
OUTPUT_MONTHLY = Path('outputs/trimmed_away_excess_contributions_by_factor.csv')

FRED_API_KEY = '51f3ac7bc8b65cb6bb2589fc570292be'
FRED_SERIES_ID = 'PCEPI'
FRED_OBSERVATION_START = '1979-12-01'

for name, path in INPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing required input {name}: {path}')

print('Inputs found:')
for name, path in INPUTS.items():
    print(f'  {name}: {path}')
print(f'  official_headline: FRED {FRED_SERIES_ID}')


Inputs found:
  trimmed_away: outputs\trimmed_away_components.csv
  classifications: output\shapiro_supercore_decomp\pce_shapiro_line_classifications.csv
  trimmed_rates: outputs\trimmed_mean_rates.csv
  component_audit: outputs\component_monthly_audit.csv
  official_headline: FRED PCEPI


In [2]:
trimmed_away = pd.read_csv(INPUTS['trimmed_away'], parse_dates=['date'])
classifications = pd.read_csv(INPUTS['classifications'], parse_dates=['date'])
trimmed_rates = pd.read_csv(INPUTS['trimmed_rates'], parse_dates=['date'])
component_audit = pd.read_csv(INPUTS['component_audit'], parse_dates=['date'])

for name, df in {
    'trimmed_away': trimmed_away,
    'classifications': classifications,
    'trimmed_rates': trimmed_rates,
    'component_audit': component_audit,
}.items():
    print(f'{name:16s} rows={len(df):>7,d} date_range={df["date"].min().date()} to {df["date"].max().date()}')


trimmed_away     rows= 65,536 date_range=1980-01-01 to 2026-04-01
classifications  rows=101,104 date_range=1979-01-01 to 2026-04-01
trimmed_rates    rows=    556 date_range=1980-01-01 to 2026-04-01
component_audit  rows= 98,968 date_range=1980-01-01 to 2026-04-01


In [3]:
def fetch_fred_series(series_id, api_key, observation_start):
    params = urllib.parse.urlencode({
        'series_id': series_id,
        'api_key': api_key,
        'file_type': 'json',
        'observation_start': observation_start,
    })
    url = f'https://api.stlouisfed.org/fred/series/observations?{params}'
    with urllib.request.urlopen(url, timeout=30) as response:
        payload = json.load(response)

    if 'error_code' in payload:
        raise RuntimeError(f"FRED error {payload.get('error_code')}: {payload.get('error_message')}")
    if 'observations' not in payload:
        raise RuntimeError(f'Unexpected FRED response keys: {sorted(payload)}')

    fred = pd.DataFrame(payload['observations'])
    fred['date'] = pd.to_datetime(fred['date'])
    fred['value'] = pd.to_numeric(fred['value'].replace('.', np.nan), errors='coerce')
    fred = fred.dropna(subset=['value']).sort_values('date').reset_index(drop=True)
    return fred[['date', 'value']].rename(columns={'value': f'{series_id.lower()}_index'})

official_pcepi = fetch_fred_series(FRED_SERIES_ID, FRED_API_KEY, FRED_OBSERVATION_START)
official_headline = official_pcepi.copy()
official_headline['official_headline_monthly_change'] = official_headline[f'{FRED_SERIES_ID.lower()}_index'].pct_change()
official_headline = official_headline.rename(columns={f'{FRED_SERIES_ID.lower()}_index': 'official_pcepi_index'})

print(f'FRED {FRED_SERIES_ID} observations: {len(official_headline):,}')
print(f'FRED {FRED_SERIES_ID} date range: {official_headline["date"].min().date()} to {official_headline["date"].max().date()}')
print(official_headline.tail().to_string(index=False))


FRED PCEPI observations: 557
FRED PCEPI date range: 1979-12-01 to 2026-04-01
      date  official_pcepi_index  official_headline_monthly_change
2025-12-01               128.576                          0.003309
2026-01-01               129.002                          0.003313
2026-02-01               129.520                          0.004015
2026-03-01               130.381                          0.006648
2026-04-01               130.902                          0.003996


In [4]:
required_trimmed_columns = {
    'date', 'period', 'line_item', 'price_change', 'trimmed_weight'
}
missing_trimmed_columns = required_trimmed_columns.difference(trimmed_away.columns)
if missing_trimmed_columns:
    raise ValueError(f'trimmed_away_components.csv is missing columns: {sorted(missing_trimmed_columns)}')

classification_lookup = classifications[['date', 'line', 'classification']].copy()
if classification_lookup.duplicated(['date', 'line']).any():
    duplicate_count = int(classification_lookup.duplicated(['date', 'line']).sum())
    raise ValueError(f'Classification file has {duplicate_count} duplicate date/line keys')

classified = trimmed_away.merge(
    classification_lookup,
    how='left',
    left_on=['date', 'line_item'],
    right_on=['date', 'line'],
    validate='many_to_one',
).drop(columns=['line'])

classified = classified.merge(
    trimmed_rates[['date', 'trimmed_monthly_change']],
    how='left',
    on='date',
    validate='many_to_one',
)

classified['excess_contribution'] = (
    classified['trimmed_weight']
    * (classified['price_change'] - classified['trimmed_monthly_change'])
)

missing_classifications = int(classified['classification'].isna().sum())
print('Classification counts:')
print(classified['classification'].value_counts(dropna=False).to_string())
print(f'\nMissing classifications: {missing_classifications:,}')


Classification counts:
classification
supply       34797
demand       17258
ambiguous    13481

Missing classifications: 0


In [5]:
classified_factor_columns = ['demand_gap', 'supply_gap', 'ambiguous_gap']
official_factor_columns = [*classified_factor_columns, 'coverage_residual_gap']
classification_to_gap = {
    'demand': 'demand_gap',
    'supply': 'supply_gap',
    'ambiguous': 'ambiguous_gap',
}

factor_monthly = (
    classified
    .groupby(['date', 'classification'], observed=True)['excess_contribution']
    .sum()
    .unstack('classification', fill_value=0.0)
    .rename(columns=classification_to_gap)
    .reset_index()
)

for column in classified_factor_columns:
    if column not in factor_monthly.columns:
        factor_monthly[column] = 0.0

factor_monthly = factor_monthly[['date', *classified_factor_columns]]
factor_monthly['total_gap'] = factor_monthly[classified_factor_columns].sum(axis=1)

component_headline_monthly = (
    component_audit
    .assign(weighted_price_change=lambda df: df['raw_weight'] * df['price_change'])
    .groupby('date', as_index=False)['weighted_price_change']
    .sum()
    .rename(columns={'weighted_price_change': 'component_headline_monthly_change'})
)

monthly = (
    trimmed_rates[['date', 'period', 'trimmed_monthly_change']]
    .merge(factor_monthly, how='left', on='date', validate='one_to_one')
    .merge(component_headline_monthly, how='left', on='date', validate='one_to_one')
    .merge(official_headline[['date', 'official_pcepi_index', 'official_headline_monthly_change']], how='left', on='date', validate='one_to_one')
    .sort_values('date')
    .reset_index(drop=True)
)
monthly[[*classified_factor_columns, 'total_gap']] = monthly[[*classified_factor_columns, 'total_gap']].fillna(0.0)
monthly['coverage_residual_gap'] = (
    monthly['official_headline_monthly_change']
    - monthly['component_headline_monthly_change']
)
monthly['official_total_gap'] = monthly['total_gap'] + monthly['coverage_residual_gap']
monthly['implied_component_headline_monthly_change'] = monthly['trimmed_monthly_change'] + monthly['total_gap']
monthly['implied_official_headline_monthly_change'] = monthly['trimmed_monthly_change'] + monthly['official_total_gap']

component_monthly_identity_residual = (
    monthly['component_headline_monthly_change']
    - monthly['trimmed_monthly_change']
    - monthly['total_gap']
)
official_monthly_identity_residual = (
    monthly['official_headline_monthly_change']
    - monthly['trimmed_monthly_change']
    - monthly['official_total_gap']
)
coverage_reconciliation_residual = monthly['official_total_gap'] - monthly['total_gap'] - monthly['coverage_residual_gap']

print(monthly.head().to_string(index=False))
print(f'\nComponent-universe monthly identity max abs residual: {component_monthly_identity_residual.abs().max():.3e}')
print(f'Official monthly identity max abs residual: {official_monthly_identity_residual.abs().max():.3e}')
print(f'Coverage reconciliation max abs residual: {coverage_reconciliation_residual.abs().max():.3e}')


      date  period  trimmed_monthly_change  demand_gap  supply_gap  ambiguous_gap  total_gap  component_headline_monthly_change  official_pcepi_index  official_headline_monthly_change  coverage_residual_gap  official_total_gap  implied_component_headline_monthly_change  implied_official_headline_monthly_change
1980-01-01 1980-01                0.007837    0.000840    0.002918      -0.001136   0.002622                           0.010459                37.124                          0.010479               0.000020            0.002643                                   0.010459                                  0.010479
1980-02-01 1980-02                0.008055    0.002343    0.000521      -0.000091   0.002773                           0.010827                37.526                          0.010829               0.000001            0.002774                                   0.010827                                  0.010829
1980-03-01 1980-03                0.007568    0.003037    0.0016

In [6]:
def compound_rate(monthly_path):
    monthly_path = np.asarray(monthly_path, dtype=float)
    return float(np.prod(1.0 + monthly_path) - 1.0)

for column in [*official_factor_columns, 'total_gap', 'official_total_gap']:
    monthly[f'{column}_12m_rolling'] = monthly[column].rolling(window=12, min_periods=12).sum()

shapley_output_columns = {
    'demand_gap': 'demand_gap_12m_shapley',
    'supply_gap': 'supply_gap_12m_shapley',
    'ambiguous_gap': 'ambiguous_gap_12m_shapley',
    'coverage_residual_gap': 'coverage_residual_gap_12m_shapley',
}
for column in shapley_output_columns.values():
    monthly[column] = np.nan
monthly['total_gap_12m_shapley'] = np.nan
monthly['official_total_gap_12m_shapley'] = np.nan
monthly['component_headline_12m_change'] = np.nan
monthly['official_headline_12m_change'] = np.nan
monthly['trimmed_12m_change'] = np.nan

permutation_count = math.factorial(len(official_factor_columns))
for end_pos in range(11, len(monthly)):
    window = monthly.iloc[end_pos - 11:end_pos + 1]
    trimmed_path = window['trimmed_monthly_change'].to_numpy(dtype=float)
    factor_paths = {
        factor: window[factor].to_numpy(dtype=float)
        for factor in official_factor_columns
    }

    component_path = trimmed_path + sum(factor_paths[factor] for factor in classified_factor_columns)
    official_path = trimmed_path + sum(factor_paths.values())
    trimmed_12m = compound_rate(trimmed_path)
    component_headline_12m = compound_rate(component_path)
    official_headline_12m = compound_rate(official_path)

    allocations = dict.fromkeys(official_factor_columns, 0.0)
    for order in permutations(official_factor_columns):
        active_gap_path = np.zeros(12, dtype=float)
        previous_value = compound_rate(trimmed_path + active_gap_path)
        for factor in order:
            active_gap_path = active_gap_path + factor_paths[factor]
            current_value = compound_rate(trimmed_path + active_gap_path)
            allocations[factor] += current_value - previous_value
            previous_value = current_value

    for factor, output_column in shapley_output_columns.items():
        monthly.at[end_pos, output_column] = allocations[factor] / permutation_count
    monthly.at[end_pos, 'total_gap_12m_shapley'] = sum(allocations[factor] for factor in classified_factor_columns) / permutation_count
    monthly.at[end_pos, 'official_total_gap_12m_shapley'] = sum(allocations.values()) / permutation_count
    monthly.at[end_pos, 'component_headline_12m_change'] = component_headline_12m
    monthly.at[end_pos, 'official_headline_12m_change'] = official_headline_12m
    monthly.at[end_pos, 'trimmed_12m_change'] = trimmed_12m

official_shapley_identity_residual = (
    monthly['official_headline_12m_change']
    - monthly['trimmed_12m_change']
    - monthly['official_total_gap_12m_shapley']
)
shapley_sum_residual = (
    monthly['official_total_gap_12m_shapley']
    - monthly[[*shapley_output_columns.values()]].sum(axis=1)
)

print(monthly.tail()[[
    'date',
    'demand_gap_12m_rolling', 'supply_gap_12m_rolling', 'ambiguous_gap_12m_rolling', 'coverage_residual_gap_12m_rolling', 'official_total_gap_12m_rolling',
    'demand_gap_12m_shapley', 'supply_gap_12m_shapley', 'ambiguous_gap_12m_shapley', 'coverage_residual_gap_12m_shapley', 'official_total_gap_12m_shapley',
    'official_headline_12m_change', 'trimmed_12m_change',
]].to_string(index=False))
print(f'\nOfficial 12-month Shapley identity max abs residual: {official_shapley_identity_residual.dropna().abs().max():.3e}')
print(f'Shapley factor-sum max abs residual: {shapley_sum_residual.dropna().abs().max():.3e}')


      date  demand_gap_12m_rolling  supply_gap_12m_rolling  ambiguous_gap_12m_rolling  coverage_residual_gap_12m_rolling  official_total_gap_12m_rolling  demand_gap_12m_shapley  supply_gap_12m_shapley  ambiguous_gap_12m_shapley  coverage_residual_gap_12m_shapley  official_total_gap_12m_shapley  official_headline_12m_change  trimmed_12m_change
2025-12-01                0.002826                0.003155                  -0.001664                          -0.000005                        0.004312                0.002894                0.003230                  -0.001706                          -0.000005                        0.004412                      0.028781            0.024369
2026-01-01                0.004021                0.002462                  -0.002141                           0.000030                        0.004372                0.004116                0.002520                  -0.002194                           0.000031                        0.004473                

In [7]:
pp_helper_columns = [
    'demand_gap_12m_rolling',
    'supply_gap_12m_rolling',
    'ambiguous_gap_12m_rolling',
    'coverage_residual_gap_12m_rolling',
    'total_gap_12m_rolling',
    'official_total_gap_12m_rolling',
    'demand_gap_12m_shapley',
    'supply_gap_12m_shapley',
    'ambiguous_gap_12m_shapley',
    'coverage_residual_gap_12m_shapley',
    'total_gap_12m_shapley',
    'official_total_gap_12m_shapley',
    'component_headline_12m_change',
    'official_headline_12m_change',
    'trimmed_12m_change',
]
for column in pp_helper_columns:
    monthly[f'{column}_pp'] = monthly[column] * 100.0

monthly_output_columns = [
    'date', 'period',
    'demand_gap', 'supply_gap', 'ambiguous_gap', 'total_gap',
    'coverage_residual_gap', 'official_total_gap',
    'trimmed_monthly_change', 'component_headline_monthly_change', 'official_headline_monthly_change', 'official_pcepi_index',
    'implied_component_headline_monthly_change', 'implied_official_headline_monthly_change',
    'demand_gap_12m_rolling', 'supply_gap_12m_rolling', 'ambiguous_gap_12m_rolling', 'coverage_residual_gap_12m_rolling', 'total_gap_12m_rolling', 'official_total_gap_12m_rolling',
    'demand_gap_12m_shapley', 'supply_gap_12m_shapley', 'ambiguous_gap_12m_shapley', 'coverage_residual_gap_12m_shapley', 'total_gap_12m_shapley', 'official_total_gap_12m_shapley',
    'component_headline_12m_change', 'official_headline_12m_change', 'trimmed_12m_change',
    *[f'{column}_pp' for column in pp_helper_columns],
]
monthly = monthly[monthly_output_columns]

print('Monthly output columns:')
print('\n'.join(monthly.columns))


Monthly output columns:
date
period
demand_gap
supply_gap
ambiguous_gap
total_gap
coverage_residual_gap
official_total_gap
trimmed_monthly_change
component_headline_monthly_change
official_headline_monthly_change
official_pcepi_index
implied_component_headline_monthly_change
implied_official_headline_monthly_change
demand_gap_12m_rolling
supply_gap_12m_rolling
ambiguous_gap_12m_rolling
coverage_residual_gap_12m_rolling
total_gap_12m_rolling
official_total_gap_12m_rolling
demand_gap_12m_shapley
supply_gap_12m_shapley
ambiguous_gap_12m_shapley
coverage_residual_gap_12m_shapley
total_gap_12m_shapley
official_total_gap_12m_shapley
component_headline_12m_change
official_headline_12m_change
trimmed_12m_change
demand_gap_12m_rolling_pp
supply_gap_12m_rolling_pp
ambiguous_gap_12m_rolling_pp
coverage_residual_gap_12m_rolling_pp
total_gap_12m_rolling_pp
official_total_gap_12m_rolling_pp
demand_gap_12m_shapley_pp
supply_gap_12m_shapley_pp
ambiguous_gap_12m_shapley_pp
coverage_residual_gap_12m_sha

In [8]:
TOLERANCE = 1e-10

classification_values = set(classified['classification'].dropna().unique())
unexpected_classifications = classification_values.difference({'demand', 'supply', 'ambiguous'})
missing_official_headline = int(monthly['official_headline_monthly_change'].isna().sum())
component_monthly_identity_max_abs_residual = float(component_monthly_identity_residual.abs().max())
official_monthly_identity_max_abs_residual = float(official_monthly_identity_residual.abs().max())
coverage_reconciliation_max_abs_residual = float(coverage_reconciliation_residual.abs().max())
component_implied_max_abs_residual = float((monthly['component_headline_monthly_change'] - monthly['implied_component_headline_monthly_change']).abs().max())
official_implied_max_abs_residual = float((monthly['official_headline_monthly_change'] - monthly['implied_official_headline_monthly_change']).abs().max())
official_shapley_identity_max_abs_residual = float(official_shapley_identity_residual.dropna().abs().max())
shapley_sum_max_abs_residual = float(shapley_sum_residual.dropna().abs().max())

assert missing_classifications == 0
assert missing_official_headline == 0
assert not unexpected_classifications, sorted(unexpected_classifications)
assert component_monthly_identity_max_abs_residual < TOLERANCE
assert official_monthly_identity_max_abs_residual < TOLERANCE
assert coverage_reconciliation_max_abs_residual < TOLERANCE
assert component_implied_max_abs_residual < TOLERANCE
assert official_implied_max_abs_residual < TOLERANCE
assert official_shapley_identity_max_abs_residual < TOLERANCE
assert shapley_sum_max_abs_residual < TOLERANCE

validation_summary = pd.DataFrame({
    'check': [
        'missing trimmed-away classifications',
        'missing official headline observations',
        'component-universe monthly identity max abs residual',
        'official monthly identity max abs residual',
        'coverage reconciliation max abs residual',
        'component headline implied max abs residual',
        'official headline implied max abs residual',
        'official 12-month Shapley identity max abs residual',
        'official Shapley factor-sum max abs residual',
    ],
    'value': [
        missing_classifications,
        missing_official_headline,
        component_monthly_identity_max_abs_residual,
        official_monthly_identity_max_abs_residual,
        coverage_reconciliation_max_abs_residual,
        component_implied_max_abs_residual,
        official_implied_max_abs_residual,
        official_shapley_identity_max_abs_residual,
        shapley_sum_max_abs_residual,
    ],
})
print(validation_summary.to_string(index=False))


                                               check        value
                missing trimmed-away classifications 0.000000e+00
              missing official headline observations 0.000000e+00
component-universe monthly identity max abs residual 2.155394e-16
          official monthly identity max abs residual 2.155394e-16
            coverage reconciliation max abs residual 8.673617e-19
         component headline implied max abs residual 2.159731e-16
          official headline implied max abs residual 2.151057e-16
 official 12-month Shapley identity max abs residual 0.000000e+00
        official Shapley factor-sum max abs residual 3.469447e-18


In [9]:
OUTPUT_CLASSIFIED.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_MONTHLY.parent.mkdir(parents=True, exist_ok=True)

classified.to_csv(OUTPUT_CLASSIFIED, index=False)
monthly.to_csv(OUTPUT_MONTHLY, index=False)

print(f'Wrote {OUTPUT_CLASSIFIED} ({len(classified):,} rows)')
print(f'Wrote {OUTPUT_MONTHLY} ({len(monthly):,} rows)')
print('\nLatest monthly rows:')
print(monthly.tail(3).to_string(index=False))


Wrote outputs\trimmed_away_components_classified.csv (65,536 rows)
Wrote outputs\trimmed_away_excess_contributions_by_factor.csv (556 rows)

Latest monthly rows:
      date  period  demand_gap  supply_gap  ambiguous_gap  total_gap  coverage_residual_gap  official_total_gap  trimmed_monthly_change  component_headline_monthly_change  official_headline_monthly_change  official_pcepi_index  implied_component_headline_monthly_change  implied_official_headline_monthly_change  demand_gap_12m_rolling  supply_gap_12m_rolling  ambiguous_gap_12m_rolling  coverage_residual_gap_12m_rolling  total_gap_12m_rolling  official_total_gap_12m_rolling  demand_gap_12m_shapley  supply_gap_12m_shapley  ambiguous_gap_12m_shapley  coverage_residual_gap_12m_shapley  total_gap_12m_shapley  official_total_gap_12m_shapley  component_headline_12m_change  official_headline_12m_change  trimmed_12m_change  demand_gap_12m_rolling_pp  supply_gap_12m_rolling_pp  ambiguous_gap_12m_rolling_pp  coverage_residual_gap_12m_roll